In [20]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

In [21]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

In [22]:
TABLES = Path("../04_outputs/tables")
FIGURES = Path("../04_outputs/figures")

TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

In [23]:
baseline_ml = pd.read_csv(TABLES / "baseline_ml_results.csv")
banglabert_binary = pd.read_csv(TABLES / "banglabert_binary_summary.csv")
fgm_ben = pd.read_csv(TABLES / "banglabert_fgm_ben_sarc_binary_results.csv")
fgm_bs3 = pd.read_csv(TABLES / "banglabert_fgm_banglasarc3_binary_results.csv")
ternary_plain = pd.read_csv(TABLES / "banglabert_banglasarc3_ternary_results.csv")
ternary_weighted = pd.read_csv(TABLES / "banglabert_weighted_banglasarc3_ternary_results.csv")
cross_plain = pd.read_csv(TABLES / "cross_dataset_comparison.csv")
cross_fgm = pd.read_csv(TABLES / "cross_fgm_banglasarc3_binary_to_ben_sarc_binary_results.csv")

In [24]:
ml_binary = baseline_ml[[
    "dataset", "test_accuracy", "test_macro_f1", "test_f1_binary"
]].copy()
ml_binary["model"] = "tfidf_logreg"

bert_binary_test = banglabert_binary[banglabert_binary["split"] == "test"].copy()
bert_binary_test = bert_binary_test[[
    "dataset", "accuracy", "macro_f1", "f1_binary"
]].rename(columns={
    "accuracy": "test_accuracy",
    "macro_f1": "test_macro_f1",
    "f1_binary": "test_f1_binary",
})
bert_binary_test["model"] = "banglabert"

fgm_binary = pd.concat([fgm_ben, fgm_bs3], ignore_index=True)
fgm_binary = fgm_binary[fgm_binary["split"] == "test"].copy()
fgm_binary = fgm_binary[[
    "dataset", "accuracy", "macro_f1", "f1_binary"
]].rename(columns={
    "accuracy": "test_accuracy",
    "macro_f1": "test_macro_f1",
    "f1_binary": "test_f1_binary",
})
fgm_binary["model"] = "banglabert_fgm"

In [25]:
binary_master = pd.concat(
    [ml_binary, bert_binary_test, fgm_binary],
    ignore_index=True
)

binary_master = binary_master[[
    "dataset", "model", "test_accuracy", "test_macro_f1", "test_f1_binary"
]].sort_values(["dataset", "model"]).reset_index(drop=True)

binary_master

,dataset,model,test_accuracy,test_macro_f1,test_f1_binary
0,banglasarc3_binary,banglabert,0.735661,0.735290,0.745192
1,banglasarc3_binary,banglabert_fgm,0.745636,0.745444,0.752427
2,banglasarc3_binary,tfidf_logreg,0.670823,0.670692,0.664122
3,banglasarc_binary,banglabert,0.976562,0.975052,0.968912
4,banglasarc_binary,tfidf_logreg,0.888672,0.880616,0.849604
5,ben_sarc_binary,banglabert,0.796412,0.795731,0.783940
6,ben_sarc_binary,banglabert_fgm,0.809672,0.809611,0.806195
7,ben_sarc_binary,tfidf_logreg,0.664587,0.664583,0.663537


In [26]:
binary_master.to_csv(TABLES / "master_binary_comparison.csv", index=False)
print(TABLES / "master_binary_comparison.csv")

../04_outputs/tables/master_binary_comparison.csv


In [27]:
plain_t = ternary_plain[ternary_plain["split"] == "test"].copy()
weighted_t = ternary_weighted[ternary_weighted["split"] == "test"].copy()

ternary_comparison = pd.DataFrame([
    {
        "dataset": "banglasarc3_ternary",
        "model": "banglabert",
        "test_accuracy": float(plain_t["accuracy"].iloc[0]),
        "test_macro_f1": float(plain_t["macro_f1"].iloc[0]),
        "test_weighted_f1": float(plain_t["weighted_f1"].iloc[0]),
    },
    {
        "dataset": "banglasarc3_ternary",
        "model": "banglabert_weighted",
        "test_accuracy": float(weighted_t["accuracy"].iloc[0]),
        "test_macro_f1": float(weighted_t["macro_f1"].iloc[0]),
        "test_weighted_f1": float(weighted_t["weighted_f1"].iloc[0]),
    },
])

ternary_comparison["macro_f1_gain_vs_plain"] = (
    ternary_comparison["test_macro_f1"]
    - ternary_comparison.loc[ternary_comparison["model"] == "banglabert", "test_macro_f1"].iloc[0]
)

ternary_comparison

,dataset,model,test_accuracy,test_macro_f1,test_weighted_f1,macro_f1_gain_vs_plain
0,banglasarc3_ternary,banglabert,0.641556,0.641263,0.641402,0.000000
1,banglasarc3_ternary,banglabert_weighted,0.652318,0.652876,0.652981,0.011613


In [28]:
ternary_comparison.to_csv(TABLES / "master_ternary_comparison.csv", index=False)
print(TABLES / "master_ternary_comparison.csv")

../04_outputs/tables/master_ternary_comparison.csv


In [29]:
cross_fgm_clean = cross_fgm.copy()
cross_fgm_clean["model"] = "banglabert_fgm"

cross_plain_clean = cross_plain.copy()
cross_plain_clean["model"] = "banglabert"

In [30]:
cross_master = pd.concat(
    [
        cross_plain_clean[[
            "source_dataset", "target_dataset", "accuracy", "macro_f1", "f1_binary",
            "in_domain_accuracy", "in_domain_macro_f1",
            "macro_f1_generalization_gap", "accuracy_generalization_gap", "model"
        ]],
        cross_fgm_clean[[
            "source_dataset", "target_dataset", "accuracy", "macro_f1", "f1_binary", "model"
        ]]
    ],
    ignore_index=True,
    sort=False
)

cross_master

,source_dataset,target_dataset,accuracy,macro_f1,f1_binary,in_domain_accuracy,in_domain_macro_f1,macro_f1_generalization_gap,accuracy_generalization_gap,model
0,ben_sarc_binary,banglasarc3_binary,0.669576,0.668127,0.646195,0.735661,0.735290,0.067164,0.066085,banglabert
1,banglasarc3_binary,ben_sarc_binary,0.685257,0.684958,0.694665,0.796412,0.795731,0.110773,0.111154,banglabert
2,banglasarc3_binary,ben_sarc_binary,0.659126,0.656113,0.688302,NaN,NaN,NaN,NaN,banglabert_fgm


In [31]:
cross_master.to_csv(TABLES / "master_cross_dataset_comparison.csv", index=False)
print(TABLES / "master_cross_dataset_comparison.csv")

../04_outputs/tables/master_cross_dataset_comparison.csv


In [32]:
fgm_ablation = pd.DataFrame([
    {
        "dataset": "ben_sarc_binary",
        "banglabert_macro_f1": 0.795731,
        "banglabert_fgm_macro_f1": 0.809611,
    },
    {
        "dataset": "banglasarc3_binary",
        "banglabert_macro_f1": 0.735290,
        "banglabert_fgm_macro_f1": 0.745444,
    },
])

fgm_ablation["fgm_gain"] = (
    fgm_ablation["banglabert_fgm_macro_f1"] - fgm_ablation["banglabert_macro_f1"]
)

fgm_ablation

,dataset,banglabert_macro_f1,banglabert_fgm_macro_f1,fgm_gain
0,ben_sarc_binary,0.795731,0.809611,0.013880
1,banglasarc3_binary,0.735290,0.745444,0.010154


In [33]:
fgm_ablation.to_csv(TABLES / "fgm_ablation_summary.csv", index=False)
print(TABLES / "fgm_ablation_summary.csv")

../04_outputs/tables/fgm_ablation_summary.csv


In [34]:
def brier_score_binary(y_true, y_prob):
    y_true = np.asarray(y_true).astype(float)
    y_prob = np.asarray(y_prob).astype(float)
    return np.mean((y_prob - y_true) ** 2)

def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if np.sum(mask) == 0:
            continue

        bin_confidence = np.mean(y_prob[mask])
        bin_accuracy = np.mean(y_true[mask])
        ece += (np.sum(mask) / len(y_true)) * abs(bin_accuracy - bin_confidence)

    return ece

In [35]:
calibration_template = pd.DataFrame([
    {"dataset": "ben_sarc_binary", "model": "banglabert", "brier_score": np.nan, "ece": np.nan},
    {"dataset": "ben_sarc_binary", "model": "banglabert_fgm", "brier_score": np.nan, "ece": np.nan},
    {"dataset": "banglasarc3_binary", "model": "banglabert", "brier_score": np.nan, "ece": np.nan},
    {"dataset": "banglasarc3_binary", "model": "banglabert_fgm", "brier_score": np.nan, "ece": np.nan},
])

calibration_template

,dataset,model,brier_score,ece
0,ben_sarc_binary,banglabert,NaN,NaN
1,ben_sarc_binary,banglabert_fgm,NaN,NaN
2,banglasarc3_binary,banglabert,NaN,NaN
3,banglasarc3_binary,banglabert_fgm,NaN,NaN


In [36]:
calibration_template.to_csv(TABLES / "calibration_template.csv", index=False)
print(TABLES / "calibration_template.csv")

../04_outputs/tables/calibration_template.csv


In [37]:
ablation_summary = pd.DataFrame([
    {
        "setting": "binary_in_domain_ben_sarc",
        "baseline_model": "banglabert",
        "improved_model": "banglabert_fgm",
        "baseline_macro_f1": 0.795731,
        "improved_macro_f1": 0.809611,
        "gain": 0.809611 - 0.795731,
    },
    {
        "setting": "binary_in_domain_banglasarc3",
        "baseline_model": "banglabert",
        "improved_model": "banglabert_fgm",
        "baseline_macro_f1": 0.735290,
        "improved_macro_f1": 0.745444,
        "gain": 0.745444 - 0.735290,
    },
    {
        "setting": "ternary_in_domain_banglasarc3",
        "baseline_model": "banglabert",
        "improved_model": "banglabert_weighted",
        "baseline_macro_f1": 0.641263,
        "improved_macro_f1": 0.652876,
        "gain": 0.652876 - 0.641263,
    },
    {
        "setting": "cross_dataset_bs3_to_ben",
        "baseline_model": "banglabert_cross_dataset",
        "improved_model": "banglabert_cross_dataset_fgm",
        "baseline_macro_f1": 0.684958,
        "improved_macro_f1": 0.656113,
        "gain": 0.656113 - 0.684958,
    },
])

ablation_summary

,setting,baseline_model,improved_model,baseline_macro_f1,improved_macro_f1,gain
0,binary_in_domain_ben_sarc,banglabert,banglabert_fgm,0.795731,0.809611,0.013880
1,binary_in_domain_banglasarc3,banglabert,banglabert_fgm,0.735290,0.745444,0.010154
2,ternary_in_domain_banglasarc3,banglabert,banglabert_weighted,0.641263,0.652876,0.011613
3,cross_dataset_bs3_to_ben,banglabert_cross_dataset,banglabert_cross_dataset_fgm,0.684958,0.656113,-0.028845


In [38]:
ablation_summary.to_csv(TABLES / "ablation_summary.csv", index=False)
print(TABLES / "ablation_summary.csv")

../04_outputs/tables/ablation_summary.csv
